# Full-Study Hallucination Metrics

Replicates the four pilot metrics for the full study (Claude Opus 4.7 / GPT-5.5 / Gemini 3.1
Pro × 4 prompting techniques × 807 videos):

1. **Hallucination matrix & signatures** — base counts and per-model H-distribution fingerprints (Finding i)
2. **Redistribution / JSD** — Jensen-Shannon divergence of each technique from Zero-Shot baseline (Finding ii)
3. **Accuracy vs safety decoupling** — crime classification accuracy + hallucination correlation (Finding iv)
4. **Multi-turn mitigation** — statistical test for hallucination differences single-turn vs multi-turn (Finding vi)
5. **Gemini missing-data bias** — chi-square test for whether missing labels are MAR (methodological hygiene)

**Inputs:**
- `panel_raw_judge_labels_full.csv` — LLM panel hallucination labels
- `all_triplets_cache.csv` — model outputs (for accuracy computation)

**Outputs (saved to `C:\Opeyemi\PROMPTS\WORD-COUNT-RESULT\`):**
- `hallucination_matrix_full.json`
- `hallucination_signatures_full.json`
- `redistribution_full.csv`
- `accuracy_safety_full.csv`
- `context_data_full.csv`
- `multiturn_test_full.json`
- `missing_bias_test_full.json`
- `metrics_summary_full.txt`


In [1]:
import os, json, re
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import jensenshannon

OUTPUT_DIR    = r'C:\Opeyemi\PROMPTS\EVALUATION'
TRIPLETS_PATH = os.path.join(OUTPUT_DIR, 'all_triplets_cache.csv')
PANEL_RAW     = os.path.join(OUTPUT_DIR, 'panel_raw_judge_labels_full.csv')

HALL_DIR = r'C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS'
os.makedirs(HALL_DIR, exist_ok=True)

# Outputs
HALL_MATRIX_OUT  = os.path.join(HALL_DIR, 'hallucination_matrix_full.json')
SIGNATURES_OUT   = os.path.join(HALL_DIR, 'hallucination_signatures_full.json')
REDIST_OUT       = os.path.join(HALL_DIR, 'redistribution_full.csv')
ACCURACY_OUT     = os.path.join(HALL_DIR, 'accuracy_safety_full.csv')
CONTEXT_OUT      = os.path.join(HALL_DIR, 'context_data_full.csv')
MULTITURN_OUT    = os.path.join(HALL_DIR, 'multiturn_test_full.json')
BIAS_OUT         = os.path.join(HALL_DIR, 'missing_bias_test_full.json')
SUMMARY_OUT      = os.path.join(HALL_DIR, 'metrics_summary_full.txt')

HCOLS = ['H1','H2','H3','H4','H5','H6']
H_NAMES = {
    'H1': 'SCENE_FABRICATION',
    'H2': 'CRIME_MISCLASSIFICATION',
    'H3': 'CRIME_MISSED',
    'H4': 'SEVERITY_MINIMIZATION',
    'H5': 'ENTITY_FABRICATION',
    'H6': 'PHANTOM_ACTORS',
}

# Load data
print('Loading panel labels and triplets...')
labels = pd.read_csv(PANEL_RAW)
df = pd.read_csv(TRIPLETS_PATH)
df['model_output'] = df['model_output'].fillna('').astype(str)

# Coerce H labels and filter -1 errors
for h in HCOLS:
    labels[h] = pd.to_numeric(labels[h], errors='coerce')
clean = labels[(labels[HCOLS] >= 0).all(axis=1)].copy()
print(f'  Panel: {len(labels):,} raw rows, {len(clean):,} clean rows')

# Majority-vote per (row_idx) to collapse cross-judge votes into one verdict per triplet
def majority(col):
    return col.mode().iloc[0] if not col.mode().empty else 0

mv = clean.groupby(['row_idx','model','technique','video','crime_type'])[HCOLS].agg(majority).reset_index()
print(f'  Triplets with majority verdicts: {len(mv):,}')

# Identify multi-turn techniques (Sequential, Least-to-Most, ReAct are multi-turn; Zero-Shot is single-turn)
MULTITURN_TECHS = {'Sequential', 'Least-to-Most', 'ReAct'}
SINGLETURN_TECHS = {'Zero-Shot'}
mv['is_multiturn'] = mv['technique'].isin(MULTITURN_TECHS).astype(int)

print(f'\nPer (model, technique) majority-vote triplet counts:')
print(mv.groupby(["model","technique"]).size().unstack(fill_value=0))


Loading panel labels and triplets...
  Panel: 19,360 raw rows, 19,358 clean rows
  Triplets with majority verdicts: 9,680

Per (model, technique) majority-vote triplet counts:
technique  Least-to-Most  ReAct  Sequential  Zero-Shot
model                                                 
Claude               807    807         804        807
GPT                  807    807         807        807
Gemini               806    807         807        807


---
## 1. Hallucination matrix & signatures (Finding i)

In [2]:
# Hallucination matrix: count of each H type per (model, technique)
print('=== HALLUCINATION MATRIX ===\n')

hm = {}
for model in mv['model'].unique():
    hm[model] = {}
    for tech in mv['technique'].unique():
        sub = mv[(mv['model']==model) & (mv['technique']==tech)]
        if len(sub) == 0:
            continue
        cell = {}
        for h in HCOLS:
            cell[H_NAMES[h]] = int(sub[h].sum())
        hm[model][tech] = cell

with open(HALL_MATRIX_OUT, 'w') as f:
    json.dump(hm, f, indent=2)
print(f'Saved: {HALL_MATRIX_OUT}\n')

# Pretty-print
print(f'{"":>8}{"":>16}', end='')
for h in HCOLS:
    print(f'{h:>5}', end='')
print(f'  {"total":>6}')
print('-' * 70)
for model, techs in hm.items():
    for tech, cell in techs.items():
        total = sum(cell.values())
        print(f'{model:>8}{tech:>16}', end='')
        for h in HCOLS:
            print(f'{cell[H_NAMES[h]]:>5}', end='')
        print(f'  {total:>6}')

# Signatures: per-model normalized H distribution (the "hallucination signature")
print('\n\n=== HALLUCINATION SIGNATURES (per-model H distribution) ===\n')

sigs = {}
for model in mv['model'].unique():
    sub = mv[mv['model']==model]
    counts = {h: int(sub[h].sum()) for h in HCOLS}
    total = sum(counts.values())
    pcts = {h: round(counts[h] / total * 100, 2) if total else 0 for h in HCOLS}
    sigs[model] = {
        'total': total,
        'counts': counts,
        'percentages': pcts,
        'n_triplets': int(len(sub)),
    }

with open(SIGNATURES_OUT, 'w') as f:
    json.dump(sigs, f, indent=2)
print(f'Saved: {SIGNATURES_OUT}\n')

# Print signatures
print(f'{"Model":<10}{"Total":>7}{"H1%":>7}{"H2%":>7}{"H3%":>7}{"H4%":>7}{"H5%":>7}{"H6%":>7}')
print('-' * 60)
for model, s in sigs.items():
    p = s['percentages']
    print(f'{model:<10}{s["total"]:>7}{p["H1"]:>7.1f}{p["H2"]:>7.1f}{p["H3"]:>7.1f}'
          f'{p["H4"]:>7.1f}{p["H5"]:>7.1f}{p["H6"]:>7.1f}')

# Identify each model's dominant H types (top-2 by percentage)
print('\nDominant H types per model:')
for model, s in sigs.items():
    p = s['percentages']
    sorted_h = sorted(p.items(), key=lambda x: -x[1])
    top2 = sorted_h[:2]
    print(f'  {model}: {top2[0][0]}={top2[0][1]:.1f}%, {top2[1][0]}={top2[1][1]:.1f}%')


=== HALLUCINATION MATRIX ===

Saved: C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS\hallucination_matrix_full.json

                           H1   H2   H3   H4   H5   H6   total
----------------------------------------------------------------------
  Claude       Zero-Shot  784  282  216   97  782  399    2560
  Claude      Sequential  712  269  207   77  686  293    2244
  Claude   Least-to-Most  619  192  327   65  612  363    2178
  Claude           ReAct  784  202  299  143  782  337    2547
     GPT       Zero-Shot  230  158  155  157  285   52    1037
     GPT      Sequential  257  128  267  286  341   87    1366
     GPT   Least-to-Most  214   16  630  638  194   23    1715
     GPT           ReAct  140    1  674   26  120   14     975
  Gemini       Zero-Shot  360   89  286  292  495  143    1665
  Gemini      Sequential  399  188  132  112  578  257    1666
  Gemini   Least-to-Most  416  136  286  234  493  181    1746
  Gemini           ReAct  367  126  369  299  471  145    1777


---
## 2. Redistribution / JSD (Finding ii)

For each (model, technique≠Zero-Shot), compute Jensen-Shannon divergence between
the technique's H-distribution and the model's Zero-Shot H-distribution. Higher JSD
means the technique redistributes failure modes (vs Zero-Shot baseline) rather than
preserving them.

In [3]:
def normalize_dist(counts):
    """Convert raw H counts to a probability distribution (with epsilon to avoid 0)."""
    arr = np.array([counts[h] for h in HCOLS], dtype=float)
    s = arr.sum()
    if s == 0:
        # If no halls, return uniform (so JSD with another all-zero is 0)
        return np.ones(6) / 6
    # Add small epsilon to avoid divide-by-zero when one component is 0
    arr = arr + 1e-9
    return arr / arr.sum()

# Build per-(model, technique) H count vectors
cell_counts = {}
for model in mv['model'].unique():
    for tech in mv['technique'].unique():
        sub = mv[(mv['model']==model) & (mv['technique']==tech)]
        if len(sub) == 0:
            continue
        cell_counts[(model, tech)] = {h: int(sub[h].sum()) for h in HCOLS}

# Compute JSD vs Zero-Shot for each (model, non-Zero-Shot)
redist_rows = []
for model in mv['model'].unique():
    zero_key = (model, 'Zero-Shot')
    if zero_key not in cell_counts:
        continue
    zero_dist = normalize_dist(cell_counts[zero_key])
    zero_total = sum(cell_counts[zero_key].values())

    for tech in mv['technique'].unique():
        key = (model, tech)
        if key not in cell_counts or tech == 'Zero-Shot':
            continue
        tech_dist = normalize_dist(cell_counts[key])
        tech_total = sum(cell_counts[key].values())
        # scipy's jensenshannon returns sqrt of JS divergence; squaring gives the JS divergence
        jsd = jensenshannon(tech_dist, zero_dist, base=2) ** 2
        redist_rows.append({
            'model': model,
            'technique': tech,
            'total': tech_total,
            'delta_from_zero': tech_total - zero_total,
            'jsd_from_zero': float(jsd),
        })

redist_df = pd.DataFrame(redist_rows).sort_values('jsd_from_zero', ascending=False).reset_index(drop=True)
redist_df.to_csv(REDIST_OUT, index=False)
print(f'Saved: {REDIST_OUT}\n')

print('=== REDISTRIBUTION: JSD from Zero-Shot baseline ===\n')
print(f'{"Model":<10}{"Technique":<16}{"Total halls":>13}{"Delta vs ZS":>13}{"JSD":>10}')
print('-' * 65)
for _, r in redist_df.iterrows():
    print(f'{r["model"]:<10}{r["technique"]:<16}{r["total"]:>13}{r["delta_from_zero"]:>+13}'
          f'{r["jsd_from_zero"]:>10.4f}')

# Headline: do techniques redistribute (high JSD) without reducing total (small delta)?
print('\nMean total halls Zero-Shot:',
      np.mean([sum(cell_counts[(m,"Zero-Shot")].values()) for m in mv['model'].unique()
               if (m,"Zero-Shot") in cell_counts]))
print('Mean total halls multi-turn:',
      redist_df['total'].mean())
print('Mean JSD from Zero-Shot:', redist_df['jsd_from_zero'].mean())


Saved: C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS\redistribution_full.csv

=== REDISTRIBUTION: JSD from Zero-Shot baseline ===

Model     Technique         Total halls  Delta vs ZS       JSD
-----------------------------------------------------------------
GPT       ReAct                     975          -62    0.2787
GPT       Least-to-Most            1715         +678    0.1671
Gemini    Sequential               1666           +1    0.0474
GPT       Sequential               1366         +329    0.0127
Claude    Least-to-Most            2178         -382    0.0088
Claude    ReAct                    2547          -13    0.0058
Gemini    Least-to-Most            1746          +81    0.0048
Gemini    ReAct                    1777         +112    0.0030
Claude    Sequential               2244         -316    0.0012

Mean total halls Zero-Shot: 1754.0
Mean total halls multi-turn: 1801.5555555555557
Mean JSD from Zero-Shot: 0.05881468964664714


---
## 3. Accuracy vs safety decoupling (Finding iv)

Accuracy = did the model correctly identify the crime type? We check whether the
ground-truth crime type (e.g. "Robbery") appears in the model's final answer.

Safety score = -1 × number of hallucinations per triplet (more halls = less safe).

Per (model, technique), report mean accuracy and total hallucinations, then compute
Spearman correlation across all 12 cells. If they're independent (low |ρ| or high p),
that's the "statistically decoupled" finding.

In [4]:
# Accuracy: does the model_output mention the correct crime_type?
def check_accuracy(crime_type, model_output):
    if not isinstance(model_output, str) or not model_output:
        return False
    # Case-insensitive substring match
    return crime_type.lower() in model_output.lower()

df['correct_crime'] = df.apply(lambda r: check_accuracy(r['crime_type'], r['model_output']), axis=1)

# Merge with the majority-vote labels to get hallucination counts per triplet
mv_with_acc = mv.merge(
    df[['model','technique','video','correct_crime']],
    on=['model','technique','video'],
    how='left',
)
mv_with_acc['hall_count'] = mv_with_acc[HCOLS].sum(axis=1)

acc_rows = []
for (m, t), grp in mv_with_acc.groupby(['model','technique']):
    acc = grp['correct_crime'].mean() * 100
    n_halls = int(grp['hall_count'].sum())
    n_h4 = int(grp['H4'].sum())
    acc_rows.append({
        'model': m, 'technique': t,
        'n_triplets': len(grp),
        'accuracy': round(acc, 2),
        'n_hallucinations': n_halls,
        'n_h4': n_h4,
        'safety_score': -n_halls,
    })

acc_df = pd.DataFrame(acc_rows).sort_values(['model','technique'])
acc_df.to_csv(ACCURACY_OUT, index=False)
print(f'Saved: {ACCURACY_OUT}\n')

print('=== ACCURACY vs HALLUCINATIONS ===\n')
print(f'{"Model":<10}{"Technique":<16}{"n":>5}{"Acc%":>8}{"Halls":>7}{"Safety":>8}')
print('-' * 56)
for _, r in acc_df.iterrows():
    print(f'{r["model"]:<10}{r["technique"]:<16}{r["n_triplets"]:>5,}{r["accuracy"]:>8.1f}'
          f'{r["n_hallucinations"]:>7}{r["safety_score"]:>8}')

# Decoupling test: Spearman correlation between accuracy and hallucination count across cells
rho_acc, p_acc = stats.spearmanr(acc_df['accuracy'], acc_df['n_hallucinations'])
print(f'\nSpearman(accuracy, hallucination count): rho={rho_acc:+.3f}  p={p_acc:.3f}  n={len(acc_df)}')
if abs(rho_acc) < 0.3 or p_acc > 0.05:
    print('-> Accuracy and hallucination rate appear STATISTICALLY DECOUPLED.')
else:
    print('-> Accuracy and hallucination rate are statistically associated.')


Saved: C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS\accuracy_safety_full.csv

=== ACCURACY vs HALLUCINATIONS ===

Model     Technique           n    Acc%  Halls  Safety
--------------------------------------------------------
Claude    Least-to-Most     807    63.0   2178   -2178
Claude    ReAct             807    79.3   2547   -2547
Claude    Sequential        804    78.9   2244   -2244
Claude    Zero-Shot         807    75.5   2560   -2560
GPT       Least-to-Most     807    58.0   1715   -1715
GPT       ReAct             807   100.0    975    -975
GPT       Sequential        807    80.0   1366   -1366
GPT       Zero-Shot         807    73.0   1037   -1037
Gemini    Least-to-Most     806    54.7   1746   -1746
Gemini    ReAct             807    61.0   1777   -1777
Gemini    Sequential        807    49.1   1666   -1666
Gemini    Zero-Shot         807    41.9   1665   -1665

Spearman(accuracy, hallucination count): rho=+0.007  p=0.983  n=12
-> Accuracy and hallucination rate appear STATISTI

---
## 4. Multi-turn vs single-turn mitigation test (Finding vi)

Tests whether multi-turn techniques (Sequential, Least-to-Most, ReAct) produce fewer
hallucinations than single-turn (Zero-Shot). Uses Mann-Whitney U on per-triplet
hallucination counts.

In [5]:
# Per-triplet hallucination count
mt_data = mv.copy()
mt_data['hall_count'] = mt_data[HCOLS].sum(axis=1)

multiturn_halls = mt_data[mt_data['is_multiturn']==1]['hall_count'].values
singleturn_halls = mt_data[mt_data['is_multiturn']==0]['hall_count'].values

print(f'Multi-turn triplets   : n={len(multiturn_halls):,}, '
      f'mean={multiturn_halls.mean():.3f}, median={np.median(multiturn_halls):.0f}')
print(f'Single-turn triplets  : n={len(singleturn_halls):,}, '
      f'mean={singleturn_halls.mean():.3f}, median={np.median(singleturn_halls):.0f}')

# Mann-Whitney U (non-parametric, doesn't assume normality)
mw_stat, mw_p = stats.mannwhitneyu(multiturn_halls, singleturn_halls, alternative='less')
print(f'\nMann-Whitney U (multiturn < singleturn): U={mw_stat:.0f}  p={mw_p:.4f}')

# Two-sided for completeness
mw_stat_2s, mw_p_2s = stats.mannwhitneyu(multiturn_halls, singleturn_halls, alternative='two-sided')
print(f'Mann-Whitney U (two-sided)              : U={mw_stat_2s:.0f}  p={mw_p_2s:.4f}')

# T-test as well (parametric)
t_stat, t_p = stats.ttest_ind(multiturn_halls, singleturn_halls, equal_var=False)
print(f'Welch t-test (two-sided)                : t={t_stat:.3f}  p={t_p:.4f}')

# Effect size (Cliff's delta)
def cliffs_delta(x, y):
    nx, ny = len(x), len(y)
    greater = sum(1 for xi in x for yj in y if xi > yj)
    less = sum(1 for xi in x for yj in y if xi < yj)
    return (greater - less) / (nx * ny)

# Cliff's delta is O(n*m) so subsample for speed if needed
if len(multiturn_halls) * len(singleturn_halls) > 5_000_000:
    rng = np.random.default_rng(42)
    mt_s = rng.choice(multiturn_halls, size=2000, replace=False)
    st_s = rng.choice(singleturn_halls, size=min(2000, len(singleturn_halls)), replace=False)
    cd = cliffs_delta(mt_s, st_s)
    print(f"Cliff's delta (subsampled, n=2000 each): {cd:.4f}")
else:
    cd = cliffs_delta(multiturn_halls, singleturn_halls)
    print(f"Cliff's delta: {cd:.4f}")

multiturn_test = {
    'multiturn_techniques': sorted(MULTITURN_TECHS),
    'singleturn_techniques': sorted(SINGLETURN_TECHS),
    'multiturn_n': int(len(multiturn_halls)),
    'singleturn_n': int(len(singleturn_halls)),
    'multiturn_mean': float(multiturn_halls.mean()),
    'singleturn_mean': float(singleturn_halls.mean()),
    'mannwhitney_one_sided': {'U': float(mw_stat), 'p': float(mw_p),
                                'alternative': 'multiturn < singleturn'},
    'mannwhitney_two_sided': {'U': float(mw_stat_2s), 'p': float(mw_p_2s)},
    'welch_t_test': {'t': float(t_stat), 'p': float(t_p)},
    'cliffs_delta': float(cd),
}
with open(MULTITURN_OUT, 'w') as f:
    json.dump(multiturn_test, f, indent=2)
print(f'\nSaved: {MULTITURN_OUT}')

# Also save context_data_full for compatibility with pilot pipeline
ctx_rows = []
for (m, t), grp in mt_data.groupby(['model','technique']):
    sub = df[(df['model']==m) & (df['technique']==t)]
    ctx_rows.append({
        'model': m, 'technique': t,
        'mean_words_per_output': float(sub['model_output'].str.split().str.len().mean()),
        'total_hallucinations': int(grp['hall_count'].sum()),
        'n_h4': int(grp['H4'].sum()),
        'is_multiturn': int(t in MULTITURN_TECHS),
    })
ctx_df = pd.DataFrame(ctx_rows).sort_values(['model','technique'])
ctx_df.to_csv(CONTEXT_OUT, index=False)
print(f'Saved: {CONTEXT_OUT}')


Multi-turn triplets   : n=7,259, mean=2.234, median=2
Single-turn triplets  : n=2,421, mean=2.173, median=2

Mann-Whitney U (multiturn < singleturn): U=8942744  p=0.9092
Mann-Whitney U (two-sided)              : U=8942744  p=0.1816
Welch t-test (two-sided)                : t=1.702  p=0.0887
Cliff's delta (subsampled, n=2000 each): 0.0090

Saved: C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS\multiturn_test_full.json
Saved: C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS\context_data_full.csv


---
## 5. Gemini missing-data bias test

In [6]:
# Were Gemini judge calls that returned ERROR or PROHIBITED_CONTENT distributed
# uniformly across (model, technique, crime_type), or biased toward specific cells?
#
# We compare the distribution of "Gemini missing" rows vs "Gemini valid" rows along
# each axis using chi-square test.

# Identify Gemini judging rows
gemini_rows = labels[labels['judge'] == 'Gemini'].copy()
gemini_rows['is_missing'] = (gemini_rows[HCOLS] < 0).any(axis=1).astype(int)
print(f'Gemini judge rows: {len(gemini_rows):,}')
print(f'  Missing (any -1):  {int(gemini_rows["is_missing"].sum())}')
print(f'  Valid:             {int(len(gemini_rows) - gemini_rows["is_missing"].sum())}')

bias_results = {}
for axis in ['model', 'technique', 'crime_type']:
    if axis not in gemini_rows.columns:
        bias_results[axis] = {'note': f'column "{axis}" not in panel labels — skipped'}
        continue
    ct = pd.crosstab(gemini_rows[axis], gemini_rows['is_missing'])
    if ct.shape[1] < 2 or ct.values.sum() == 0:
        bias_results[axis] = {'note': 'insufficient missing data to test'}
        continue
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    bias_results[axis] = {
        'chi2': float(chi2),
        'p': float(p),
        'dof': int(dof),
        'observed': ct.to_dict(),
    }
    print(f'\n=== Gemini missingness by {axis} ===')
    print(ct)
    print(f'  chi2={chi2:.3f}, p={p:.4f}, dof={dof}')

with open(BIAS_OUT, 'w') as f:
    json.dump(bias_results, f, indent=2, default=str)
print(f'\nSaved: {BIAS_OUT}')


Gemini judge rows: 6,453
  Missing (any -1):  2
  Valid:             6451

=== Gemini missingness by model ===
is_missing     0  1
model              
Claude      3224  1
GPT         3227  1
  chi2=0.000, p=1.0000, dof=1

=== Gemini missingness by technique ===
is_missing        0  1
technique             
Least-to-Most  1614  0
ReAct          1612  2
Sequential     1611  0
Zero-Shot      1614  0
  chi2=5.998, p=0.1117, dof=3

=== Gemini missingness by crime_type ===
is_missing        0  1
crime_type            
Abuse           397  2
Assault          88  0
Burglary        800  0
Explosion       399  0
Fighting        399  0
RoadAccidents  1184  0
Robbery        1192  0
Shooting        400  0
Shoplifting     400  0
Stealing        800  0
Vandalism       392  0
  chi2=30.355, p=0.0007, dof=10

Saved: C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS\missing_bias_test_full.json


---
## 6. Paper-ready summary

In [7]:
lines = []
lines.append('=' * 76)
lines.append('FULL-STUDY HALLUCINATION METRICS — SUMMARY')
lines.append('=' * 76)
lines.append('')

# Signatures
lines.append('-- HALLUCINATION SIGNATURES (per-model H distribution) --')
for model, s in sigs.items():
    p = s['percentages']
    lines.append(f'  {model:<10} total halls={s["total"]:>5}  H1={p["H1"]:>5.1f}%  '
                 f'H2={p["H2"]:>5.1f}%  H3={p["H3"]:>5.1f}%  H4={p["H4"]:>5.1f}%  '
                 f'H5={p["H5"]:>5.1f}%  H6={p["H6"]:>5.1f}%')
lines.append('')

# Redistribution
lines.append('-- REDISTRIBUTION (JSD vs Zero-Shot, top 5) --')
for _, r in redist_df.head(5).iterrows():
    lines.append(f'  {r["model"]:<10}{r["technique"]:<16} JSD={r["jsd_from_zero"]:.4f}  '
                 f'delta={r["delta_from_zero"]:+d}')
lines.append('')

# Accuracy decoupling
lines.append('-- ACCURACY vs SAFETY DECOUPLING --')
lines.append(f'  Spearman(accuracy, hallucinations) across n={len(acc_df)} cells: '
             f'rho={rho_acc:+.3f}  p={p_acc:.3f}')
if abs(rho_acc) < 0.3 or p_acc > 0.05:
    lines.append('  Conclusion: accuracy and hallucination rate are STATISTICALLY DECOUPLED.')
else:
    lines.append('  Conclusion: accuracy and hallucination rate are associated.')
lines.append('')

# Multi-turn test
lines.append('-- MULTI-TURN MITIGATION (Mann-Whitney U) --')
lines.append(f'  multi-turn n={multiturn_test["multiturn_n"]:,}  '
             f'mean halls/triplet={multiturn_test["multiturn_mean"]:.3f}')
lines.append(f'  single-turn n={multiturn_test["singleturn_n"]:,}  '
             f'mean halls/triplet={multiturn_test["singleturn_mean"]:.3f}')
lines.append(f'  Mann-Whitney U (one-sided, multiturn<singleturn): U='
             f'{multiturn_test["mannwhitney_one_sided"]["U"]:.0f}  '
             f'p={multiturn_test["mannwhitney_one_sided"]["p"]:.4f}')
lines.append(f"  Cliff's delta: {multiturn_test['cliffs_delta']:+.4f}")
lines.append('')

# Bias test
lines.append('-- GEMINI MISSING-DATA BIAS (chi-square) --')
for axis, res in bias_results.items():
    if 'note' in res:
        lines.append(f'  {axis}: {res["note"]}')
    else:
        lines.append(f'  {axis}: chi2={res["chi2"]:.3f}, p={res["p"]:.4f}, dof={res["dof"]}')

summary = '\n'.join(lines)
with open(SUMMARY_OUT, 'w') as f:
    f.write(summary)
print(summary)
print(f'\nSaved: {SUMMARY_OUT}')


FULL-STUDY HALLUCINATION METRICS — SUMMARY

-- HALLUCINATION SIGNATURES (per-model H distribution) --
  Claude     total halls= 9529  H1= 30.4%  H2=  9.9%  H3= 11.0%  H4=  4.0%  H5= 30.0%  H6= 14.6%
  GPT        total halls= 5093  H1= 16.5%  H2=  6.0%  H3= 33.9%  H4= 21.7%  H5= 18.5%  H6=  3.5%
  Gemini     total halls= 6854  H1= 22.5%  H2=  7.9%  H3= 15.7%  H4= 13.7%  H5= 29.7%  H6= 10.6%

-- REDISTRIBUTION (JSD vs Zero-Shot, top 5) --
  GPT       ReAct            JSD=0.2787  delta=-62
  GPT       Least-to-Most    JSD=0.1671  delta=+678
  Gemini    Sequential       JSD=0.0474  delta=+1
  GPT       Sequential       JSD=0.0127  delta=+329
  Claude    Least-to-Most    JSD=0.0088  delta=-382

-- ACCURACY vs SAFETY DECOUPLING --
  Spearman(accuracy, hallucinations) across n=12 cells: rho=+0.007  p=0.983
  Conclusion: accuracy and hallucination rate are STATISTICALLY DECOUPLED.

-- MULTI-TURN MITIGATION (Mann-Whitney U) --
  multi-turn n=7,259  mean halls/triplet=2.234
  single-turn n=2,421